In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.listdir('/content/drive/MyDrive/Maple-leafs-project/')

Mounted at /content/drive


['tor_2025.csv',
 'tor_2024.csv',
 'tor_2024.gsheet',
 'tor_2025.gsheet',
 'goalies2024-25.csv',
 'goalies2025-26.csv',
 'tor_games.db',
 'leafs_games.csv',
 'leafs_goalies.csv']

In [2]:
import pandas as pd, sqlite3
folder = '/content/drive/MyDrive/Maple-leafs-project/'

# Team files (really Excel files, despite the .csv name)
df24 = pd.read_excel(folder + 'tor_2024.csv', engine='openpyxl')
df25 = pd.read_excel(folder + 'tor_2025.csv', engine='openpyxl')
df24['season'] = '2024-25'
df25['season'] = '2025-26'
games = pd.concat([df24, df25])

# Goalie files (real CSVs)
g24 = pd.read_csv(folder + 'goalies2024-25.csv')
g25 = pd.read_csv(folder + 'goalies2025-26.csv')

# Save all tables, then close the connection
conn = sqlite3.connect(folder + 'tor_games.db')
games.to_sql('games', conn, if_exists='replace', index=False)
g24.to_sql('goalies_2024', conn, if_exists='replace', index=False)
g25.to_sql('goalies_2025', conn, if_exists='replace', index=False)
conn.close()

print(len(games), len(g24), len(g25))

820 515 490


In [3]:
%pip install -q jupysql
%load_ext sql
%sql sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db
%config SqlMagic.displaylimit = 100

Connecting to 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

In [4]:
##Did everything load correctly?

%%sql
SELECT season, situation, COUNT(*) AS games
FROM games
GROUP BY season, situation

Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

season,situation,games
2024-25,4on5,82
2024-25,5on4,82
2024-25,5on5,82
2024-25,all,82
2024-25,other,82
2025-26,4on5,82
2025-26,5on4,82
2025-26,5on5,82
2025-26,all,82
2025-26,other,82


In [5]:
## Goals for and against per season

%%sql
SELECT season,
  SUM(goalsFor) AS goals_for,
  SUM(goalsAgainst) AS goals_against,
  SUM(goalsFor) - SUM(goalsAgainst) AS goal_diff
FROM games
WHERE situation = 'all'
GROUP BY season

Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

season,goals_for,goals_against,goal_diff
2024-25,267,229,38
2025-26,252,295,-43


In [6]:
## 5 on 5 shots per 60 minutes

%%sql
SELECT season,
  ROUND(SUM(iceTime) / 3600.0, 1) AS hours_5v5,
  ROUND(SUM(shotsOnGoalFor) * 3600.0 / SUM(iceTime), 2) AS shots_for_per60,
  ROUND(SUM(shotsOnGoalAgainst) * 3600.0 / SUM(iceTime), 2) AS shots_against_per60
FROM games
WHERE situation = '5on5'
GROUP BY season

Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

season,hours_5v5,shots_for_per60,shots_against_per60
2024-25,67.7,26.46,27.25
2025-26,68.7,24.25,30.05


In [7]:
## Shot quality and goaltending

%%sql
SELECT season,
  ROUND(SUM(xGoalsAgainst) * 3600.0 / SUM(iceTime), 2) AS xga_per60,
  ROUND(SUM(goalsAgainst) * 3600.0 / SUM(iceTime), 2) AS ga_per60,
  ROUND(SUM(xGoalsAgainst) - SUM(goalsAgainst), 2) AS gsax
FROM games
WHERE situation = '5on5'
GROUP BY season

Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

season,xga_per60,ga_per60,gsax
2024-25,2.39,2.01,26.14
2025-26,2.67,3.01,-23.35


In [8]:
## Before vs After March 30 (the day they fired their GM)

%%sql
SELECT
  CASE WHEN gameDate < 20260330 THEN 'before'ELSE 'after' END AS period,
  COUNT(*) AS games,
  ROUND(SUM(shotsOnGoalAgainst) * 3600.0 / SUM(iceTime), 2) AS sa_per60,
  ROUND(SUM(xGoalsAgainst) * 3600.0 / SUM(iceTime) ,2) AS xga_per60,
  ROUND(SUM(goalsAgainst) * 3600.0 / SUM(iceTime), 2) AS ga_per60,
  ROUND(SUM(xGoalsAgainst) - SUM(goalsAgainst), 2) AS gsax,
  ROUND(SUM(xGoalsFor) * 3600.0 / SUM(iceTime),2) AS xgf_per60,
  ROUND(SUM(goalsFor) * 3600.0 / SUM(iceTime), 2) AS gf_per60
FROM games
WHERE situation = '5on5' AND season = '2025-26'
GROUP BY period


Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

period,games,sa_per60,xga_per60,ga_per60,gsax,xgf_per60,gf_per60
after,8,27.84,2.87,3.72,-5.74,2.04,1.79
before,74,30.29,2.65,2.94,-17.61,2.27,2.57


Finding: The drop started before March 30 (GSAx was already −17.6 in the first 74 games), so tanking doesn’t explain the collapse. The last 8 games show scoring fell too, but the sample is small.

In [9]:
## Running total game by game, 2025-26, 5-on-5

%%sql

SELECT gameDate,
  opposingTeam,
  goalsAgainst,
  ROUND(xGoalsAgainst, 2) AS xga,
  SUM(goalsAgainst) OVER (ORDER BY gameDate) AS running_ga,
  ROUND(SUM(xGoalsAgainst) OVER (ORDER BY gameDate), 1) AS running_xga,
  ROUND(SUM(xGoalsAgainst - goalsAgainst) OVER (ORDER BY gameDate), 1) AS running_gsax
FROM games
WHERE season = '2025-26' AND situation = '5on5'
ORDER BY gameDate

Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

gameDate,opposingTeam,goalsAgainst,xga,running_ga,running_xga,running_gsax
20251008,MTL,1,2.44,1,2.4,1.4
20251011,DET,3,2.35,4,4.8,0.8
20251013,DET,2,1.24,6,6.0,0.0
20251014,NSH,4,1.71,10,7.7,-2.3
20251016,NYR,1,1.83,11,9.6,-1.4
20251018,SEA,2,1.28,13,10.8,-2.1
20251021,NJD,3,1.69,16,12.5,-3.5
20251024,BUF,4,1.91,20,14.4,-5.6
20251025,BUF,2,1.04,22,15.5,-6.5
20251028,CGY,3,2.84,25,18.3,-6.7


In [10]:
## Each Leafs goalie: GSAx 2024-25 vs 2025-26, 5-on-5

%%sql
SELECT a.name,
  ROUND(a.xGoals - a.goals, 1) AS gsax_2024,
  ROUND(b.xGoals - b.goals, 1) AS gsax_2025,
  ROUND(b.xGoals - b.goals - (a.xGoals - a.goals), 1) AS change
FROM goalies_2024 a
JOIN goalies_2025 b
  ON a.playerId = b.playerId AND a.situation = b.situation
WHERE a.team = 'TOR' AND b.team = 'TOR' AND a.situation = '5on5'
ORDER BY change

Running query in 'sqlite:////content/drive/MyDrive/Maple-leafs-project/tor_games.db'

name,gsax_2024,gsax_2025,change
Anthony Stolarz,24.2,-10.6,-34.7
Joseph Woll,7.4,-8.8,-16.2
Dennis Hildeby,-3.8,3.7,7.5


Finding: Anthony Stolarz had the biggest drop, from 24.2 GSAx in 2024-25 (3rd best in the NHL) to -10.6 GSAx in 2025-2026. Together, Stolarz and Woll fell by about 51 goals, roughly the team's entire goaltending decline. This make goaltending the main reason for Leaf's 30 point drop. Also Stolarz missed about two months with an injury starting in November 2025 and never regained his form, which likely explains much of his decline.


In [11]:
## Clean files for Tableau

games.to_csv(folder + 'leafs_games.csv', index=False)

goalies = pd.concat([g24.assign(season='2024-25'), g25.assign(season='2025-26')])

goalies = goalies[goalies['team'] == 'TOR']
goalies.to_csv(folder + 'leafs_goalies.csv', index=False)

print(len(games), len(goalies))


820 45
